In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import shapiro, levene, ttest_ind, kruskal
from statsmodels.stats.multitest import multipletests

DATA_FILE = "Water Quality.csv"
LOCATION_COLUMN = "MonitoringLocationID"
PARAMETER_COLUMN = "CharacteristicName"
VALUE_COLUMN = "ResultValue"
OUTPUT_DIR = "task3_outputs"
ALPHA = 0.05

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(DATA_FILE, low_memory=False)
required = [LOCATION_COLUMN, PARAMETER_COLUMN, VALUE_COLUMN]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nAvailable columns: {list(df.columns)}")

df[VALUE_COLUMN] = pd.to_numeric(df[VALUE_COLUMN], errors="coerce")
df = df.dropna(subset=[LOCATION_COLUMN, PARAMETER_COLUMN, VALUE_COLUMN]).copy()

results = []
assumption_results = []

for parameter, part in df.groupby(PARAMETER_COLUMN):
    location_counts = part[LOCATION_COLUMN].value_counts()
    eligible_locations = location_counts[location_counts >= 2]
    if len(eligible_locations) < 2:
        continue

    # Compare the two locations with the most measurements for this parameter.
    loc1, loc2 = eligible_locations.index[:2]
    a = part.loc[part[LOCATION_COLUMN] == loc1, VALUE_COLUMN].astype(float).values
    b = part.loc[part[LOCATION_COLUMN] == loc2, VALUE_COLUMN].astype(float).values

    # Shapiro-Wilk tests for normality; sample at most 5,000 records per group.
    normality_p = []
    for location, values in [(loc1, a), (loc2, b)]:
        sample = values
        if len(sample) > 5000:
            sample = np.random.default_rng(42).choice(sample, size=5000, replace=False)
        if len(sample) >= 3:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                p_norm = shapiro(sample).pvalue
        else:
            p_norm = np.nan
        normality_p.append(p_norm)
        assumption_results.append({
            "Parameter": parameter,
            "Location": location,
            "Sample_Size": len(values),
            "Shapiro_Wilk_p_value": p_norm,
            "Potential_non_normality": bool(p_norm < ALPHA) if pd.notna(p_norm) else None
        })

    lev_p = levene(a, b, center="median").pvalue
    t_stat, t_p = ttest_ind(a, b, equal_var=False)

    # Kruskal-Wallis compares rank distributions across all eligible locations.
    all_groups = [
        part.loc[part[LOCATION_COLUMN] == loc, VALUE_COLUMN].astype(float).values
        for loc in eligible_locations.index
    ]
    if len(all_groups) >= 2:
        try:
            kw_stat, kw_p = kruskal(*all_groups)
        except ValueError:
            kw_stat, kw_p = np.nan, np.nan
    else:
        kw_stat, kw_p = np.nan, np.nan

    results.append({
        "Parameter": parameter,
        "Location_1": loc1,
        "Location_2": loc2,
        "n_1": len(a),
        "n_2": len(b),
        "Mean_1": np.mean(a),
        "Mean_2": np.mean(b),
        "Median_1": np.median(a),
        "Median_2": np.median(b),
        "Welch_t_statistic": t_stat,
        "Welch_p_value": t_p,
        "Levene_p_value": lev_p,
        "Shapiro_p_location_1": normality_p[0],
        "Shapiro_p_location_2": normality_p[1],
        "Kruskal_Wallis_statistic_all_eligible_locations": kw_stat,
        "Kruskal_Wallis_p_value_all_eligible_locations": kw_p
    })

if not results:
    raise ValueError(
        "No parameters had at least two measurements in at least two locations. "
        "Inspect missing values or use MonitoringLocationName if appropriate."
    )

result_df = pd.DataFrame(results)
result_df["Welch_p_FDR_adjusted"] = multipletests(
    result_df["Welch_p_value"].fillna(1.0), method="fdr_bh"
)[1]
result_df["Welch_significant_FDR_0_05"] = result_df["Welch_p_FDR_adjusted"] < ALPHA

# Adjust Kruskal-Wallis p-values separately.
# Adjust Kruskal-Wallis p-values separately.

if "Kruskal_Wallis_p_value" not in result_df.columns:
    result_df["Kruskal_Wallis_p_value"] = np.nan

kw_valid = result_df["Kruskal_Wallis_p_value"].notna()

result_df["Kruskal_p_FDR_adjusted"] = np.nan

if kw_valid.any():
    result_df.loc[kw_valid, "Kruskal_p_FDR_adjusted"] = multipletests(
        result_df.loc[kw_valid, "Kruskal_Wallis_p_value"],
        method="fdr_bh"
    )[1]

result_df["Kruskal_significant_FDR_0_05"] = (
    result_df["Kruskal_p_FDR_adjusted"] < ALPHA
)

print("Hypothesis test results:")
print(result_df.round(5).to_string(index=False))
print(f"\nResults saved in: {OUTPUT_DIR}")
print(
    "\nInterpretation: Welch's t-test compares the means of the two selected "
    "locations for each parameter. Levene p < 0.05 suggests unequal variances. "
    "Shapiro-Wilk p < 0.05 suggests possible non-normality, though large samples "
    "can make this test sensitive. Kruskal-Wallis compares rank distributions "
    "across eligible locations. Adjusted p-values help account for multiple tests. "
    "These tests show evidence of differences, not causation."
)

Hypothesis test results:
                                Parameter Location_1 Location_2  n_1  n_2    Mean_1    Mean_2  Median_1  Median_2  Welch_t_statistic  Welch_p_value  Levene_p_value  Shapiro_p_location_1  Shapiro_p_location_2  Kruskal_Wallis_statistic_all_eligible_locations  Kruskal_Wallis_p_value_all_eligible_locations  Welch_p_FDR_adjusted  Welch_significant_FDR_0_05  Kruskal_Wallis_p_value  Kruskal_p_FDR_adjusted  Kruskal_significant_FDR_0_05
                        Alkalinity, total NS01ED0110 NS01ED0111  267  228   2.89876   2.22588     2.200    2.1000            4.77062        0.00000         0.00000               0.00000               0.00000                                       2224.88809                                        0.00000               0.00001                        True                     NaN                     NaN                         False
                                 Aluminum NS01ED0005 NS01ED0110  875  857 235.50354 182.15986   216.000  134.00